In [1]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic import BaseModel, Field
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

model = OpenAIChatModel(
    model_name ="nvidia/nemotron-3.5-lightning-30b-a3b",
    provider = OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1", api_key =os.environ["NVIDIA_API_KEY"]
    ),
)
model

OpenAIChatModel(model_name='nvidia/nemotron-3.5-lightning-30b-a3b')

In [2]:
ads = []

for path in sorted(Path("data").glob("ads*.txt")):
    ads.append(path.read_text(encoding="utf-8"))
len(ads)


3

In [3]:
job_agent = Agent(
    model=model,
    instructions=(
        "You summarize Swedish/English job ads. "
        "Write a short summary: role, company, key requirements and what the job involves."
    ),
)

async def summarize_ad(ad: str) -> str:
    result = await job_agent.run(ad)
    return result.output

In [4]:
summary = await summarize_ad(ads[0])
print(summary)

**Role:** Data Engineer (with Analytics Engineering focus)  
**Company:** Instabee – a fast-growing European e-commerce enabler (formed 2022 by combining Budbee & Instabox), serving millions of consumers across Europe with last-mile delivery and digital services.  
**Key Requirements:**  
- Building modern data platforms on GCP and Snowflake  
- Designing integration patterns for APIs, MongoDB, MySQL  
- Setting up CI/CD pipelines and Infrastructure as Code (IaC)  
- Orchestrating data pipelines (e.g., Airflow)  
- Data modeling with DBT/SQL and programming in Python (or Java/Go)  
- University degree in a technical/analytical field  
- Collaborative mindset: bridging business, data, and tech teams  
- Based in Stockholm, hybrid work (3 days from office)  
- Interest/experience in Analytics Engineering is a plus  

**What the Job Involves:**  
- Shape and build Instabee’s data platform, enabling better decision-making across the organization  
- Lead source integrations and migrations,

In [5]:
for i, ad in enumerate(ads, start=1):
    summary = await summarize_ad(ad)                
    with open(f"summary_{i}.md", "w", encoding="utf-8") as file:
        file.write("# Job ad\n\n" + ad + "\n\n# Summary\n\n" + summary)